Fase 5 Modul 1: SELECT, WHERE, GROUP BY, HAVING

In [2]:
from pathlib import Path
import pandas as pd
import sqlite3

# 1. cari CSV (pola lama yang sudah terbukti)
csv = Path("data/clean_tips.csv") if Path("data/clean_tips.csv").exists() else Path("../data/clean_tips.csv")
print(f"csv: {csv}")  # harusnya ../data/clean_tips.csv kalau kamu di 05-sql/
df = pd.read_csv(csv)
df = df[(df["tip"] >= 0) & (df["total_bill"] < 100)]  # 241

# 2. DB ikut folder yang sama dengan CSV (bukan "data/" mentah)
db = csv.parent / "tips.db"  # hasilnya ../data/tips.db
db.parent.mkdir(parents=True, exist_ok=True)  # jaga-jaga folder belum ada
con = sqlite3.connect(db)
df.to_sql("tips", con, if_exists="replace", index=False)
print(pd.read_sql("SELECT COUNT(*) AS n FROM tips", con))  # 241

csv: ..\data\clean_tips.csv
     n
0  241


In [3]:
# SELECT + WHERE = pilih kolom + saring baris (mirip df[kondisi])
print(pd.read_sql("SELECT day, total_bill, tip FROM tips WHERE time='Dinner' LIMIT 3", con))

# GROUP BY = rekap per grup (mirip groupby). Tanpa GROUP BY, agregat = 1 baris total.
print(pd.read_sql("""SELECT day, COUNT(*) AS n, AVG(total_bill) AS avg_bill
FROM tips GROUP BY day ORDER BY avg_bill DESC""", con))

# HAVING = saring HASIL rekap. WHERE saring baris mentah, HAVING saring grup.
print(pd.read_sql("""SELECT day, COUNT(*) AS n, AVG(tip_pct) AS avg_tip
FROM tips GROUP BY day HAVING COUNT(*) >= 20""", con))
# baca: grup Fri (n=19) hilang karena < 20. Ini cara jujur singkirkan sampel kecil.

   day  total_bill   tip
0  Sat       21.01  3.50
1  Sun       23.68  3.31
2  Sun       24.59  3.61
    day   n   avg_bill
0   Sun  73  21.627671
1   Sat  88  20.447841
2  Thur  61  17.759508
3   Fri  19  17.151579
    day   n    avg_tip
0   Sat  88  15.330439
1   Sun  73  16.846052
2  Thur  61  16.139743
